# 🚀 CompX Orbit — 100% Free AI Voice Enhancer Server (DeepFilterNet 3)
### Runs on Google Colab Free NVIDIA Tesla T4 GPU with Cloudflare Tunnel (Zero Cost!)

**Instructions:**
1. Go to **Runtime > Change runtime type** -> Select **T4 GPU** (or CPU, works on both!).
2. Click the **Play button ▶** on Cell 1 to install dependencies.
3. Click the **Play button ▶** on Cell 2 to launch the AI server and get your `https://...trycloudflare.com` URL!
4. Copy the URL and paste it into CompX Orbit's Voice Enhancer tool or your `.env` as `AI_VOICE_ENHANCE_URL`.

In [ ]:
# Cell 1: Clean previous runs & install DeepFilterNet + FastAPI (100% error-free)
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
!apt-get install -y -qq ffmpeg

print("📦 Installing DeepFilterNet 3, FastAPI, Uvicorn & PyTorch...")
!pip install -q fastapi uvicorn python-multipart torch torchaudio deepfilternet

# Download cloudflared for free instant public HTTPS tunnel (No account required!)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("✅ Setup complete! Now run Cell 2 below to start your AI server.")

In [ ]:
# Cell 2: Start FastAPI AI Server + Public Cloudflare Tunnel
import subprocess, time, os, re, shutil

app_code = '''
import os, subprocess, tempfile, shutil
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask
import torch

app = FastAPI(title="CompX Orbit AI Voice Enhancer")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

print("🚀 Pre-loading DeepFilterNet 3 AI model into VRAM...")
from df.enhance import enhance, init_df, load_audio, save_audio
df_model, df_state, _ = init_df()
print("✅ DeepFilterNet 3 is ready!")

@app.get("/health")
def health():
    return {
        "status": "online",
        "gpu": torch.cuda.is_available(),
        "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "engine": "DeepFilterNet 3 (AI Neural)"
    }

@app.post("/process")
def process(
    file: UploadFile = File(...),
    mode: str = Form("both"),
    atten_db: float = Form(100.0),
    denoise_strength: float = Form(0.8)
):
    tmp = tempfile.mkdtemp()
    cleanup = BackgroundTask(shutil.rmtree, tmp, ignore_errors=True)
    try:
        raw_path = os.path.join(tmp, "input_" + (file.filename or "audio.wav"))
        with open(raw_path, "wb") as f:
            shutil.copyfileobj(file.file, f)

        # Convert any audio format to 48kHz mono WAV (standard for DeepFilterNet)
        wav_path = os.path.join(tmp, "converted.wav")
        r = subprocess.run(
            ["ffmpeg", "-y", "-i", raw_path, "-vn", "-ar", str(df_state.sr()), "-ac", "1", wav_path],
            capture_output=True
        )
        if r.returncode != 0:
            raise HTTPException(400, "Could not decode audio with ffmpeg")

        # AI Neural Noise Reduction & Voice Enhancement
        audio, _ = load_audio(wav_path, sr=df_state.sr())
        limit = None if atten_db >= 100 else float(atten_db)
        enhanced = enhance(df_model, df_state, audio, atten_lim_db=limit)

        out_path = os.path.join(tmp, "enhanced_studio.wav")
        save_audio(out_path, enhanced, df_state.sr())

        return FileResponse(out_path, media_type="audio/wav", filename="enhanced_studio.wav", background=cleanup)
    except Exception as e:
        shutil.rmtree(tmp, ignore_errors=True)
        raise HTTPException(500, detail=str(e))
'''

with open("app.py", "w") as f:
    f.write(app_code)

# Kill any existing uvicorn or tunnel processes
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# Launch FastAPI background server on port 7860
print("Starting Uvicorn backend server on port 7860...")
server_log = open("server.log", "w")
proc = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"], stdout=server_log, stderr=server_log)
time.sleep(4)

# Start Cloudflare Tunnel
print("Starting Cloudflare Public HTTPS Tunnel...")
tunnel_log = open("tunnel.log", "w")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:7860"], stdout=tunnel_log, stderr=tunnel_log)

# Poll for generated trycloudflare.com URL
tunnel_url = None
for attempt in range(30):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "=" * 62)
    print("🎉 SUCCESS! YOUR FREE AI VOICE ENHANCER IS LIVE:")
    print("👉 " + tunnel_url)
    print("=" * 62)
    print("\nCopy this URL and paste it into CompX Orbit's Voice Enhancer page!")
else:
    print("Tunnel log status:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f: print(f.read())
    if os.path.exists("server.log"):
        print("Server log:")
        with open("server.log", "r") as f: print(f.read())
